# test_robot.ipynb — MuJoCo Physics & Controller Sanity Check

**Purpose:** Verify the robot model, physics, and base controller in isolation — no RL.

Run cells **one at a time** and confirm each test before moving on.

| Test | Expected |
|---|---|
| 1 — Load | nq=13, nv=12, nu=6; chassis Z ≈ 0.25 |
| 2 — Forward | x > 0, y ≈ 0 |
| 3 — Turn right | y < 0 |
| 4 — Spin | x ≈ 0, y ≈ 0 |
| 5 — Dead side | y_fault < y_healthy (turns right) |
| 6 — Base controller | Robot follows L-shape waypoints |

In [ ]:
%matplotlib inline
import mujoco
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

model = mujoco.MjModel.from_xml_path("assets/robot.xml")
data  = mujoco.MjData(model)

print(f"nq = {model.nq}   (expect 13: 7 freejoint + 6 hinge)")
print(f"nv = {model.nv}   (expect 12: 6 freejoint + 6 hinge)")
print(f"nu = {model.nu}   (expect 6 actuators)")

mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)
print(f"chassis Z = {data.qpos[2]:.4f}   (expect 0.2500)")
print(f"jnt_dofadr = {model.jnt_dofadr[:]}   (expect [0 6 7 8 9 10 11])")

for i in range(model.nu):
    jnt_id   = model.actuator_trnid[i, 0]
    jnt_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, jnt_id)
    act_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i)
    print(f"  ctrl[{i}] = {act_name:8s} -> {jnt_name}")

# ── shared helpers ────────────────────────────────────────────────────────────
def run_sim(ctrl_fn, n_steps=500, every=10, height=360, width=640):
    """Run simulation. ctrl_fn(step) -> 6 wheel velocities.
    Returns (frames, final_x, final_y)."""
    mujoco.mj_resetData(model, data)
    data.qpos[0:3] = [0.0, 0.0, 0.25]
    data.qpos[3:7] = [1.0, 0.0, 0.0, 0.0]
    mujoco.mj_forward(model, data)
    frames = []
    with mujoco.Renderer(model, height=height, width=width) as renderer:
        for t in range(n_steps):
            data.ctrl[:] = ctrl_fn(t)
            mujoco.mj_step(model, data)
            if t % every == 0:
                renderer.update_scene(data)
                frames.append(renderer.render().copy())
    return frames, float(data.qpos[0]), float(data.qpos[1])

def show_animation(frames, title=""):
    """Display frames as inline animation."""
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.axis("off")
    ax.set_title(title)
    im = ax.imshow(frames[0])
    def update(i):
        im.set_data(frames[i])
        return [im]
    ani = animation.FuncAnimation(
        fig, update, frames=len(frames), interval=40, blit=True)
    plt.close(fig)
    return HTML(ani.to_jshtml())

matplotlib.rcParams['animation.embed_limit'] = 100  # MB
print("\nCell 1 PASSED - helpers defined")

In [ ]:
# Test 2: Forward motion — all 6 wheels same velocity
frames, x, y = run_sim(lambda t: [10.0]*6)
print(f"Final pos: x={x:.3f}, y={y:.3f}")
print(f"Expected:  x > 0, y ~= 0")
assert x > 0.1,      f"Robot did not move forward! x={x:.3f}"
assert abs(y) < 0.3, f"Robot drifted laterally! y={y:.3f}"
print("Test 2 PASSED")
show_animation(frames, title="Test 2: Forward motion")

In [ ]:
# Test 3: Turn right — left wheels forward, right wheels stopped
frames, x, y = run_sim(lambda t: [5.0, 5.0, 5.0, 0.0, 0.0, 0.0])
print(f"Final pos: x={x:.3f}, y={y:.3f}")
print(f"Expected:  y < 0 (turned right)")
assert y < -0.02, f"Robot did not turn right! y={y:.3f}"
print("Test 3 PASSED")
show_animation(frames, title="Test 3: Turn right (left=5.0, right=0.0)")

In [ ]:
# Test 4: In-place spin — left forward, right backward
frames, x, y = run_sim(lambda t: [10.0, 10.0, 10.0, -10.0, -10.0, -10.0])
print(f"Final pos: x={x:.3f}, y={y:.3f}")
print(f"Expected:  both near 0 (spun in place)")
assert abs(x) < 0.5, f"Robot translated too much in X! x={x:.3f}"
assert abs(y) < 0.5, f"Robot translated too much in Y! y={y:.3f}"
print("Test 4 PASSED")
show_animation(frames, title="Test 4: In-place spin")

In [ ]:
# Test 5: Fault preview — right side dead vs healthy
frames_h, x0, y0 = run_sim(lambda t: [3.0]*6)
frames_f, x1, y1 = run_sim(lambda t: [10.0, 10.0, 10.0, 10.0, 10.0, 3.0])
print(f"No fault:        x={x0:.3f}, y={y0:.3f}")
print(f"Right side dead: x={x1:.3f}, y={y1:.3f}")
print(f"Expected: y_fault < y_healthy (robot turns right)")
assert y1 < y0 - 0.02, \
    f"No drift detected! y_fault={y1:.3f}, y_healthy={y0:.3f}"
print("Test 5 PASSED - fault causes visible asymmetric drift")
show_animation(frames_f, title="Test 5: Right side degraded")

In [ ]:
# Test 6: Base controller trajectory — no RL, no fault
import sys
sys.path.insert(0, ".")
from controllers.base_controller import WaypointController, BaseAllocator, WAYPOINTS

ctrl  = WaypointController()
alloc = BaseAllocator()

def controller_step():
    x, y = data.qpos[0], data.qpos[1]
    qw, qx, qy, qz = data.qpos[3:7]
    heading = np.arctan2(2*(qw*qz + qx*qy), 1 - 2*(qy**2 + qz**2))
    v, omega, done = ctrl.compute(np.array([x, y]), heading)
    return alloc.allocate(v, omega)

# Reset
mujoco.mj_resetData(model, data)
data.qpos[0:3] = [0, 0, 0.25]
data.qpos[3:7] = [1, 0, 0, 0]
ctrl.reset()

# Camera — top-down view of the 5×5 trajectory
cam = mujoco.MjvCamera()
cam.lookat[0] = 2.5
cam.lookat[1] = 2.5
cam.lookat[2] = 0.0
cam.distance  = 17.0
cam.elevation = -90
cam.azimuth   = 90

# Run simulation — collect trajectory and frames
xs, ys, frames = [], [], []
with mujoco.Renderer(model, height=480, width=640) as renderer:
    for t in range(18000):
        data.ctrl[:] = controller_step()
        mujoco.mj_step(model, data)
        xs.append(data.qpos[0])
        ys.append(data.qpos[1])
        if t % 10 == 0:
            renderer.update_scene(data, camera=cam)
            frames.append(renderer.render().copy())

# Trajectory plot
wps = np.vstack([[0, 0], WAYPOINTS])
plt.figure(figsize=(6, 6))
plt.plot(xs, ys, label="robot path")
plt.plot(wps[:, 0], wps[:, 1], 'rx--', markersize=10, label="waypoints")
plt.axis("equal"); plt.grid(True); plt.legend()
plt.xlabel("x (m)"); plt.ylabel("y (m)")
plt.title("Base controller trajectory (no RL, no fault)")
plt.show()

# Video
show_animation(frames, title="Test 6: Base controller trajectory")